In [1]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


cwd: C:\Users\okofoworola\Acme Health Demo\fine-tuning


# Lab 13 · Continuous Evaluation & Monitoring

Lab 07 scored quality **once** before release. Production drifts: data changes, models update, edge cases pile up. This lab runs an **online eval loop** over live traffic, emits a quality metric to **Application Insights**, and **alerts on drift** the moment scores fall below your bar. *Know quality dropped before your members do.*


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every model call below shows up live in the Microsoft Foundry portal under **your project → Tracing**.*


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='acme-conteval-lab')


[tracing] enabled. Service name : acme-conteval-lab
[tracing] Open Foundry portal -> your project -> Tracing tab
[tracing] (also visible under App Insights: appi-shuttervoice-dev)


True

---
## Step 1 — Sample live traffic and score it (LIVE)

We replay a slice of `data/eval_dataset.jsonl` through the live model and score each answer with a fast, deterministic rubric (keyword coverage). Swap in an LLM-judge for nuance in production.


In [3]:
import json, time
from pathlib import Path
from _advisor import try_build_client

client = try_build_client()
DEPLOY = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')

rows = []
p = Path('data/eval_dataset.jsonl')
if p.exists():
    rows = [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()][:5]
print('Sampled', len(rows), 'eval rows')

def score(answer: str, must_include: list[str]) -> float:
    if not answer:
        return 0.0
    a = answer.lower()
    return round(sum(1 for k in must_include if k.lower() in a) / max(len(must_include), 1), 2)

results = []
for r in rows:
    q = r.get('question') or r.get('prompt') or r.get('input') or ''
    keys = r.get('keywords') or r.get('must_include') or []
    if isinstance(keys, str):
        keys = [keys]
    if client is None or not q:
        ans = '[mock] refill via the Acme app or mail order'
    else:
        resp = client.chat.completions.create(model=DEPLOY, max_tokens=120,
            messages=[{'role': 'user', 'content': q}])
        ans = resp.choices[0].message.content or ''
    results.append(score(ans, keys) if keys else 1.0)
print('Per-item scores:', results)


Sampled 5 eval rows
Per-item scores: [1.0, 1.0, 1.0, 1.0, 1.0]


---
## Step 2 — Emit the quality metric + drift alert

Aggregate to a rolling quality score, **publish it as an OpenTelemetry metric** (lands in App Insights when tracing is on), and raise an alert if it breaches the SLO.


In [4]:
QUALITY_SLO = 0.70
avg = round(sum(results) / len(results), 3) if results else 0.0
print(f'Rolling quality score: {avg}  (SLO >= {QUALITY_SLO})')

# Publish as an OTel metric -> App Insights (no-op if tracing disabled).
try:
    from opentelemetry import metrics
    meter = metrics.get_meter('acme.continuous_eval')
    g = meter.create_gauge('acme.quality_score')
    g.set(avg, {'deployment': DEPLOY})
    print('Metric acme.quality_score emitted ->', DEPLOY)
except Exception as e:
    print('[metric skipped]', type(e).__name__, e)

if avg < QUALITY_SLO:
    print(f'🚨 DRIFT ALERT: quality {avg} < SLO {QUALITY_SLO} -> page on-call / block deploy.')
else:
    print('✅ Within SLO.')


Rolling quality score: 1.0  (SLO >= 0.7)
Metric acme.quality_score emitted -> gpt-4o-mini
✅ Within SLO.


---
## Step 3 — Replace-in-production: make it continuous

Run the loop on a schedule and wire an Azure Monitor alert to the metric so humans only get involved when quality actually drops.


In [5]:
MONITOR_REFERENCE = '''
# --- replace-in-production: scheduled online eval + alert ---
# 1) Package Step 1-2 as a job (Azure Container Apps job / Functions timer).
# 2) Schedule every 15 min over a fresh sample of production transcripts.
# 3) Alert on the emitted metric:
az monitor metrics alert create -g $RG -n quality-drift \\
  --scopes $APPINSIGHTS_ID \\
  --condition "avg customMetrics/acme.quality_score < 0.70" \\
  --window-size 15m --evaluation-frequency 5m \\
  --action $ACTION_GROUP_ID
# (Foundry: Evaluations -> Continuous evaluation can do this in-portal too.)
'''
print(MONITOR_REFERENCE)



# --- replace-in-production: scheduled online eval + alert ---
# 1) Package Step 1-2 as a job (Azure Container Apps job / Functions timer).
# 2) Schedule every 15 min over a fresh sample of production transcripts.
# 3) Alert on the emitted metric:
az monitor metrics alert create -g $RG -n quality-drift \
  --scopes $APPINSIGHTS_ID \
  --condition "avg customMetrics/acme.quality_score < 0.70" \
  --window-size 15m --evaluation-frequency 5m \
  --action $ACTION_GROUP_ID
# (Foundry: Evaluations -> Continuous evaluation can do this in-portal too.)



---
## Takeaways

- Quality is a **metric you watch**, not a one-time gate.
- The same scoring rubric runs offline (Lab 07) **and** online (here).
- Scores flow to **App Insights**, so dashboards + alerts come for free.
- **Drift pages you** before members feel it.

*← The Decision Advisor (Lab 09) routes the `needs_continuous_eval` flag here.*
